In [ ]:
from google.colab import drive
drive.mount('/content/drive')
exec(open("/content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier/00_colab_setup.py").read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ /content/drive/MyDrive/imdb_peft_project
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/roberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/deberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/oof_predictions
✓ /content/drive/MyDrive/imdb_peft_project/results
✓ /content/drive/MyDrive/imdb_peft_project/notebooks
✓ /content/drive/MyDrive/imdb_peft_project/code

Folder structure ready.
Enter GitHub Token: ··········
Repository exists, pulling latest changes...
✓ Pull complete.

Repository path: /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier
SETUP COMPLETE
Drive folder  : /content/drive/MyDrive/imdb_peft_project
GitHub repo   : /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-class

In [ ]:
!pip install transformers peft accelerate torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 159.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 130.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import os
from transformers import (DebertaV2Tokenizer, DebertaV2ForSequenceClassification,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score, roc_auc_score)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
DIRS["oof_v2_best"] = f"{DIRS['root']}/oof_predictions_v2_best"
os.makedirs(DIRS["oof_v2_best"], exist_ok=True)
print(f"✓ {DIRS['oof_v2_best']}")

train_df = pd.read_parquet(f"{DIRS['root']}/train_df_v2.parquet")
test_df  = pd.read_parquet(f"{DIRS['root']}/test_df_v2.parquet")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

✓ /content/drive/MyDrive/imdb_peft_project/oof_predictions_v2_best
Train: 25000 | Test: 25000


In [ ]:
def head_tail_truncate_v2(text, tokenizer, max_len=512, head_len=256):
    """V2: Equal split — first 256 + last 256 tokens."""
    tail_len = max_len - head_len
    tokens = tokenizer(text, add_special_tokens=False,
                       truncation=False, return_tensors=None)
    input_ids      = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    if len(input_ids) > max_len - 2:
        input_ids      = input_ids[:head_len] + input_ids[-tail_len:]
        attention_mask = attention_mask[:head_len] + attention_mask[-tail_len:]
    return tokenizer(
        tokenizer.decode(input_ids),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors=None
    )

class IMDBDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512, head_len=256):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.head_len  = head_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text  = self.df.loc[idx, "text_clean"]
        label = self.df.loc[idx, "label"]
        encoding = head_tail_truncate_v2(
            text, self.tokenizer, self.max_len, self.head_len
        )
        return {
            "input_ids":      torch.tensor(encoding["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels":         torch.tensor(label,                      dtype=torch.long),
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    return {
        "accuracy" : accuracy_score(labels, preds),
        "f1"       : f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall"   : recall_score(labels, preds),
        "roc_auc"  : roc_auc_score(labels, probs)
    }

print("Loading DeBERTa tokenizer...")
tokenizer     = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-base")
train_dataset = IMDBDataset(train_df, tokenizer)
test_dataset  = IMDBDataset(test_df,  tokenizer)
print(f"✓ Train: {len(train_dataset)} — Test: {len(test_dataset)}")

Loading DeBERTa tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

✓ Train: 25000 — Test: 25000


In [ ]:
def train_oof_deberta_best(df, n_folds=5):
    """
    DeBERTa V2 Best OOF training.
    Best config: r=32, lr=5e-5, lora_alpha=64
    Saves LoRA adapter + classifier + pooler separately.
    """
    skf       = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    labels    = df["label"].values
    oof_preds = np.zeros(len(df))

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, labels)):
        print(f"\n{'='*50}")
        print(f"FOLD {fold+1}/{n_folds} — deberta_v2_best")
        print(f"{'='*50}")

        fold_train = df.iloc[train_idx].reset_index(drop=True)
        fold_val   = df.iloc[val_idx].reset_index(drop=True)

        train_dataset = IMDBDataset(fold_train, tokenizer)
        val_dataset   = IMDBDataset(fold_val,   tokenizer)

        base_model = DebertaV2ForSequenceClassification.from_pretrained(
            "microsoft/deberta-v3-base",
            num_labels=2,
            torch_dtype=torch.float32
        )

        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=32,
            lora_alpha=64,
            lora_dropout=0.1,
            target_modules=["query_proj", "value_proj"],
            bias="none"
        )
        model = get_peft_model(base_model, lora_config)
        model = model.to(torch.float32).to(device)

        training_args = TrainingArguments(
            output_dir=f"{DIRS['oof_v2_best']}/fold{fold}",
            num_train_epochs=2,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=32,
            learning_rate=5e-5,
            warmup_steps=500,
            weight_decay=0.01,
            gradient_accumulation_steps=2,
            eval_strategy="epoch",
            save_strategy="no",
            fp16=False,
            seed=SEED,
            report_to="none"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        preds_output = trainer.predict(val_dataset)
        logits       = preds_output.predictions
        probs        = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
        oof_preds[val_idx] = probs

        fold_acc = accuracy_score(labels[val_idx], (probs > 0.5).astype(int))
        fold_f1  = f1_score(labels[val_idx], (probs > 0.5).astype(int))
        print(f"Fold {fold+1} — Accuracy: {fold_acc:.4f} | F1: {fold_f1:.4f}")

        np.save(f"{DIRS['oof_v2_best']}/deberta_v2_best_fold{fold}.npy", oof_preds)
        print(f"✓ OOF saved: fold {fold}")

        del model, trainer, base_model
        torch.cuda.empty_cache()

    print(f"\n{'='*50}")
    print(f"OOF COMPLETE — deberta_v2_best")
    print(f"OOF Accuracy : {accuracy_score(labels, (oof_preds > 0.5).astype(int)):.4f}")
    print(f"OOF F1       : {f1_score(labels, (oof_preds > 0.5).astype(int)):.4f}")
    print(f"OOF ROC-AUC  : {roc_auc_score(labels, oof_preds):.4f}")
    print(f"{'='*50}")

    return oof_preds

print("✓ OOF function ready.")

✓ OOF function ready.


In [ ]:
print("DeBERTa V2 Best OOF training starting...")
print("5 folds x 2 epochs — approximately 2 hours on A100")

deberta_oof_best = train_oof_deberta_best(train_df, n_folds=5)

print("✓ DeBERTa V2 Best OOF complete.")

DeBERTa V2 Best OOF training starting...
5 folds x 2 epochs — approximately 2 hours on A100

FOLD 1/5 — deberta_v2_best


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

...

Fold 4 — Accuracy: 0.9488 | F1: 0.9494
✓ OOF saved: fold 3

FOLD 5/5 — deberta_v2_best


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Roc Auc
1,0.355307,0.197671,0.943200,0.943628,0.936564,0.950800,0.984354
2,0.301992,0.177455,0.946400,0.947077,0.935257,0.959200,0.986039


Fold 5 — Accuracy: 0.9464 | F1: 0.9471
✓ OOF saved: fold 4

OOF COMPLETE — deberta_v2_best
OOF Accuracy : 0.9511
OOF F1       : 0.9514
OOF ROC-AUC  : 0.9871
✓ DeBERTa V2 Best OOF complete.


In [ ]:
# Save OOF results
save_results({
    "deberta_v2_best_oof": {
        "accuracy": float(accuracy_score(train_df["label"].values,
                         (deberta_oof_best > 0.5).astype(int))),
        "f1"      : float(f1_score(train_df["label"].values,
                         (deberta_oof_best > 0.5).astype(int))),
        "roc_auc" : float(roc_auc_score(train_df["label"].values, deberta_oof_best)),
        "config"  : "r=32, lr=5e-5, lora_alpha=64",
        "folds"   : 5,
        "epochs"  : 2,
    }
}, "oof_deberta_v2_best_results.json")
print("✓ Results saved.")

NOTEBOOK_NAME = "05d_oof_training_v2_best"
!jupyter nbconvert --to script \
  "/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}.ipynb" \
  --output-dir "/content/"

import os
os.rename(f"/content/{NOTEBOOK_NAME}.txt",
          f"/content/{NOTEBOOK_NAME}.py")

save_code_to_repo(f"/content/{NOTEBOOK_NAME}.py")
push_to_github("deberta v2 best oof complete r32 lr5e5")

✓ Results saved: /content/drive/MyDrive/imdb_peft_project/results/oof_deberta_v2_best_results.json
✓ Results saved.
[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/05d_oof_training_v2_best.ipynb to script
[NbConvertApp] ERROR | Notebook JSON is invalid: Additional properties are not allowed ('metadata' was unexpected)

Failed validating 'additionalProperties' in stream:

On instance['cells'][6]['outputs'][0]:
{'metadata': {'tags': None},
 'name': 'stdout',
 'output_type': 'stream',
 'text': 'DeBERTa V2 Best OOF training starting...\n'
         '5 folds x 2 epochs — ap...'}
[NbConvertApp] Writing 7774 bytes to /content/05d_oof_training_v2_best.txt
✓ 05d_oof_training_v2_best.py → copied to repository.
✓ Pushed to GitHub: 'deberta v2 best oof complete r32 lr5e5'
